<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/20_bar_by_bar_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys, os
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Project root:", PROJ_ROOT)
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

Project root: /content/regime-aware-ml-trading/regime-aware-ml-trading
src exists: True


In [3]:
# 20 — Bar-by-Bar Baseline and Pattern-Recency Enrichment

**Objective:** the supervisor flagged that comparing against a stratified
DummyClassifier is too weak a baseline, and requested a comparison against
a classifier trained without trading-signal-based preselection — i.e.
using every bar as a training sample rather than only detector-selected
events.

**Part 1:** plain bar-by-bar baseline (no pattern information at all).
**Part 2:** following supervisor feedback, pattern information is
reintroduced not as a binary flag but as a recency count (bars since
last detection, capped at 10) — more informative for tree-based splits
than a sparse 0/1 indicator.

SyntaxError: invalid character '—' (U+2014) (1928326302.py, line 5)

In [ ]:
from src.data.load_data import load_spy
df_yf = load_spy(source='csv')
print(f'Loaded {len(df_yf)} bars')

In [ ]:
## Part 1: Plain Bar-by-Bar Baseline (no pattern info)

In [ ]:
from src.data.utils import compute_atr

def label_every_bar(df, pt_mult=2.0, sl_mult=2.0, max_holding=10, atr_window=14):
    """Triple-barrier label EVERY bar, direction-agnostic."""
    atr = compute_atr(df, window=atr_window)
    results = []
    for i in range(len(df) - max_holding):
        entry_price = df["Close"].iloc[i]
        atr_val = atr.iloc[i]
        if pd.isna(atr_val) or atr_val <= 0:
            continue
        upper = entry_price + pt_mult * atr_val
        lower = entry_price - sl_mult * atr_val
        label = "no_trade"
        exit_price = df["Close"].iloc[i + max_holding]
        exit_date = df.index[i + max_holding]
        bars_held = max_holding
        for j in range(i + 1, i + max_holding + 1):
            high_j = df["High"].iloc[j]
            low_j = df["Low"].iloc[j]
            hit_upper = high_j >= upper
            hit_lower = low_j <= lower
            if hit_upper and hit_lower:
                exit_price = df["Close"].iloc[j]; exit_date = df.index[j]; bars_held = j - i; break
            elif hit_upper:
                label = "long"; exit_price = upper; exit_date = df.index[j]; bars_held = j - i; break
            elif hit_lower:
                label = "short"; exit_price = lower; exit_date = df.index[j]; bars_held = j - i; break
        results.append({"event_date": df.index[i], "entry_price": entry_price,
                        "exit_date": exit_date, "exit_price": exit_price,
                        "bars_held": bars_held, "label": label})
    return pd.DataFrame(results)

bar_by_bar_labeled = label_every_bar(df_yf)
print(f"Total bars labeled: {len(bar_by_bar_labeled)}")
print(bar_by_bar_labeled['label'].value_counts())
print(bar_by_bar_labeled['label'].value_counts(normalize=True) * 100)

In [ ]:
from src.features.build_features import compute_all_indicators

indicators = compute_all_indicators(df_yf)

bar_by_bar_labeled_valid = bar_by_bar_labeled[
    bar_by_bar_labeled['event_date'].isin(indicators.index)
].reset_index(drop=True)

bar_dates = pd.DatetimeIndex(bar_by_bar_labeled_valid['event_date'])
bar_features = indicators.loc[bar_dates].reset_index(drop=True)

abs_sma_cols = [c for c in bar_features.columns if c.startswith("sma_") and "_dist" not in c]
bar_features = bar_features.drop(columns=abs_sma_cols, errors="ignore")
bar_features = bar_features.dropna(axis=1, how="all")

bar_labels = bar_by_bar_labeled_valid['label']

print(f"Bar-by-bar feature matrix: {bar_features.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report

n = len(bar_features)
n_train = int(n * 0.6)
n_val = int(n * 0.2)

X_train_bar = bar_features.iloc[:n_train].fillna(0)
y_train_bar = bar_labels.iloc[:n_train]
X_test_bar = bar_features.iloc[n_train+n_val:].fillna(0)
y_test_bar = bar_labels.iloc[n_train+n_val:]

rf_bar = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
rf_bar.fit(X_train_bar, y_train_bar)
test_pred_bar = rf_bar.predict(X_test_bar)

print(f"=== Plain bar-by-bar classifier, TEST SET ===")
print(f"F1 macro: {f1_score(y_test_bar, test_pred_bar, average='macro'):.4f}")
print(classification_report(y_test_bar, test_pred_bar))

baseline_bar = DummyClassifier(strategy='stratified', random_state=42)
baseline_bar.fit(X_train_bar, y_train_bar)
baseline_pred_bar = baseline_bar.predict(X_test_bar)
print(f"Baseline F1 macro: {f1_score(y_test_bar, baseline_pred_bar, average='macro'):.4f}")

In [ ]:
## Part 2: Enriched with Pattern-Recency Features

Per supervisor feedback: instead of a binary 0/1 "pattern active" flag,
use a recency count — bars since the last detection of each pattern type,
capped at 10 — which gives tree-based splits more usable structure than
a sparse binary indicator.

In [ ]:
from src.patterns.scanner import scan_all_patterns

df_scanned = scan_all_patterns(df_yf.copy())

def bars_since_last_event(event_mask, max_val=10):
    """For each bar, count bars since the last True in event_mask, capped at max_val."""
    event_mask = event_mask.values if hasattr(event_mask, 'values') else event_mask
    result = np.full(len(event_mask), max_val, dtype=int)
    last_seen = -1
    for i in range(len(event_mask)):
        if event_mask[i]:
            last_seen = i
            result[i] = 0
        elif last_seen >= 0:
            result[i] = min(i - last_seen, max_val)
    return result

recency_features = pd.DataFrame({
    'bars_since_near_support': bars_since_last_event(df_scanned['near_support']),
    'bars_since_near_resistance': bars_since_last_event(df_scanned['near_resistance']),
    'bars_since_triangle': bars_since_last_event(df_scanned['triangle_pattern'].notna()),
    'bars_since_channel': bars_since_last_event(df_scanned['channel_pattern'].notna()),
    'bars_since_mtb': bars_since_last_event(df_scanned['multiple_top_bottom_pattern'].notna()),
}, index=df_scanned.index)

print(recency_features.describe())
print()
print(recency_features.head(20))

In [ ]:
recency_aligned = recency_features.loc[bar_dates].reset_index(drop=True)
bar_features_enriched = pd.concat([bar_features, recency_aligned], axis=1)

print(f"Enriched feature matrix: {bar_features_enriched.shape}")
print(f"New columns added: {list(recency_aligned.columns)}")

In [ ]:
X_train_e = bar_features_enriched.iloc[:n_train].fillna(0)
y_train_e = bar_labels.iloc[:n_train]
X_test_e = bar_features_enriched.iloc[n_train+n_val:].fillna(0)
y_test_e = bar_labels.iloc[n_train+n_val:]

rf_enriched = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
rf_enriched.fit(X_train_e, y_train_e)
pred_e = rf_enriched.predict(X_test_e)

print(f"=== Enriched (bar-by-bar + recency features), TEST SET ===")
print(f"F1 macro: {f1_score(y_test_e, pred_e, average='macro'):.4f}")
print(classification_report(y_test_e, pred_e))

print("\nFeature importance (top 15):")
fi = pd.DataFrame({'feature': X_train_e.columns, 'importance': rf_enriched.feature_importances_}).sort_values('importance', ascending=False)
print(fi.head(15).to_string(index=False))

In [ ]:
## Conclusion

| Approach | Training samples | Test F1 (macro) |
|---|---:|---:|
| Event-based, SPY-only | ~90 | 0.19–0.28 |
| Event-based, pooled 9-instrument | ~1,345 | 0.48–0.52 |
| Bar-by-bar, no pattern info | 2,400 | 0.461 |
| Bar-by-bar + recency features | 2,400 | [fill in from Cell 11] |

[Add interpretation once results are in — e.g. whether recency features
improved on the plain bar-by-bar baseline, and whether they rank highly
in feature importance.]